In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
import jieba 
import re
import numpy as np
load_dotenv()

True

In [2]:
kc_routes_path=os.getenv("KC_ROUTES_PATH")
kc_df=pd.read_json(kc_routes_path,orient='index')

kc_df = kc_df.rename(columns={0: 'kc_route'})
leaf_to_id = dict(zip(kc_df['kc_route'], kc_df.index))

questions_path=os.getenv("QUESTIONS_PATH")
t_df=pd.read_json(questions_path)
question_df=t_df.transpose()
question_df=question_df.reset_index()

In [ ]:
responses_path=os.getenv("RESPONSES_TRAIN_PATH")
responses_path=os.path.join(responses_path,"train_valid_sequences.csv")

training_set=pd.read_csv(responses_path)

In [ ]:
def map_kc_routes_to_ids(kc_routes_list):
    # If it's a string and contains '----', treat it as a single path
    if isinstance(kc_routes_list, str):
        # It's a single path string, convert to list with one item
        kc_routes_list = [kc_routes_list]
    elif not isinstance(kc_routes_list, list):
        return kc_routes_list
    
    # Now process each path
    ids = []
    for full_path in kc_routes_list:
        # Split by '----' and get the last part (leaf)
        leaf = full_path.split('----')[-1] if '----' in full_path else full_path
        # Map to ID, or keep original if not found
        ids.append(leaf_to_id.get(leaf, f"NOT FOUND: {leaf}"))
    return ids


question_df['kc_ids']=question_df['kc_routes'].apply(map_kc_routes_to_ids)

analysis_df=question_df[['index','analysis','kc_ids']]
analysis_df=analysis_df.rename(columns={'index':'Q_id'})


df = training_set.copy()

for col in ['questions', 'concepts', 'responses','timestamps']:
    if isinstance(df[col].iloc[0], str):
        # If stored as comma-separated string
        df[col] = df[col].str.split(',')
    # If it's already a list, keep it

# explode all columns simultaneously
df = df.explode(['questions', 'concepts', 'responses','timestamps'])

# Convert columns to appropriate types
df['questions'] = df['questions'].astype(int)
df['concepts'] = df['concepts'].astype(int)
df['responses'] = df['responses'].astype(int)

# Filter out padding responses (-1)
df = df[df['responses'] != -1]
df=df[df['questions'] != -1]
df=df[df['concepts'] != -1]

df=df.drop(['selectmasks','is_repeat'],axis=1)

#calculate error rates per question
question_stats = df.groupby('questions').agg(
    attempted=('responses', 'count'),
    correct=('responses', lambda x: (x == 1).sum()),
    wrong=('responses', lambda x: (x == 0).sum())
).reset_index()

question_stats['error_rate'] = question_stats['wrong'] / question_stats['attempted']

# Calculate error rates per KC (concept)
kc_stats = df.groupby('concepts').agg(
    attempted=('responses', 'count'),
    correct=('responses', lambda x: (x == 1).sum()),
    wrong=('responses', lambda x: (x == 0).sum())
).reset_index()

kc_stats['error_rate'] = kc_stats['wrong'] / kc_stats['attempted']

#add the metadata to the questions analysis and kc_routes
analysis_metadata=analysis_df.merge(question_stats[['questions', 'attempted', 'correct', 'wrong', 'error_rate']], 
left_on='Q_id', right_on='questions', how='left').drop('questions', axis=1)

question_metadata= question_df.merge(question_stats[['questions', 'attempted', 'correct', 'wrong', 'error_rate']], 
left_on='index', right_on='questions', how='left').drop('questions', axis=1)

kc_metadata = kc_df.reset_index().rename(columns={'index': 'kc_id'})
kc_metadata=kc_metadata.merge(kc_stats[['concepts', 'attempted', 'correct', 'wrong', 'error_rate']],
left_on='kc_id', right_on='concepts', how='left').drop('concepts', axis=1)

print("Analysis Metadata:")
print(analysis_metadata.head())
print("Question Metadata:")
print(question_metadata.head())
print("KC Metadata:")
print(kc_metadata.head())



Analysis Metadata:
   Q_id                                           analysis  kc_ids  attempted  \
0     0  小宇先选，有$$4$$种，接下来小明选，有$$3$$种，最后小丽选，有$$2$$种，所以一共...     [0]     2233.0   
1     1  先分成三类：情况 $$1$$：取的是英语和语文书时有：$$2\times 4=8$$（种）；...  [1, 2]     1472.0   
2     2  先考虑$$A$$区，有$$5$$种选择，接下来$$B$$区有$$4$$种，$$C$$区有$$...     [3]     2170.0   
3     3  第一步：给$$A$$染色，有$$4$$种颜色可选．\n第二步：给$$B$$染色，由于$$B$...     [3]     3233.0   
4     4                         $$3\times 3\times 3=27$$个．     [4]      767.0   

   correct  wrong  error_rate  
0   2023.0  210.0    0.094044  
1    952.0  520.0    0.353261  
2   1783.0  387.0    0.178341  
3   2822.0  411.0    0.127127  
4    686.0   81.0    0.105606  
Question Metadata:
   index                                            content  \
0      0  学校有舞蹈，唱歌、围棋、绘画四种兴趣班． 小宇、小明、小丽三个小朋友准备报名，每人只能报一个...   
1      1  书架上有 $$2$$ 本不同的英语书，$$4$$ 本不同的语文书，$$3$$ 本不同的数学书...   
2      2  用$$5$$种不同的颜色给下面的图形染色，要求相邻的区域（有公共边的两区域称为相邻）染成不同...   
3      3  用四种颜色去涂如图所示的三块区域，要求相邻

In [ ]:
analysis_metadata.to_csv(os.getenv("ANALYSIS_METADATA_PATH"), index=False,encoding='utf-8')
question_metadata.to_csv(os.getenv("QUESTION_METADATA_PATH"), index=False,encoding='utf-8')  
kc_metadata.to_csv(os.getenv("KC_METADATA_PATH"), index=False,encoding='utf-8')

## Textual Features of Questions and Analysis

In [2]:
question_metadata=pd.read_csv(os.getenv("QUESTION_METADATA_PATH"),)

def extract_question_features(question_df):
    """
    Extract linguistic and structural features from question text.
    
    Features:
    - question_length: number of words (with repeats allowed)
    - num_variables: count of numbers in the text (ranges count as one)
    - num_sentences: count of full stops (。.!? etc.)
    - num_clauses: count of commas (，、; etc.)
    - relies_on_image: 1 if 'image' appears in text, else 0
    - vocabulary_richness: unique vocabulary words using jieba
    - num_solutions: count of answer values
    - solution_complexity: number of separate answers required
    """
    
    # Compile regex patterns
    number_pattern = re.compile(r'\d+')  # Matches individual numbers
    range_pattern = re.compile(r'\d+\s*[-–—]\s*\d+')  # Matches ranges like "5-10"
    fullstop_pattern = re.compile(r'[。！？!?\.]')  # Sentence boundaries
    comma_pattern = re.compile(r'[，、；;]')  # Clause boundaries
    image_pattern = re.compile(r'image', re.IGNORECASE)
    
    features = []
    
    for idx, row in question_df.iterrows():
        content = str(row['content']) if pd.notna(row['content']) else ''
        
        # 1. Question length (word count with repeats)
        # Tokenize with jieba for Chinese
        words = list(jieba.cut(content))
        question_length = len(words)
        
        # 2. Count numbers (ranges count as one)
        # First remove ranges so they're counted as one
        content_no_ranges = range_pattern.sub(' RANGE ', content)
        numbers = number_pattern.findall(content_no_ranges)
        num_variables = len(numbers)
        
        # 3. Count sentences (full stops)
        num_sentences = len(fullstop_pattern.findall(content))
        
        # 4. Count clauses (commas)
        num_clauses = len(comma_pattern.findall(content))
        
        # 5. Check if relies on image
        relies_on_image = 1 if image_pattern.search(content) else 0
        
        # 6. Vocabulary richness (unique words)
        unique_words = set(jieba.cut(content))
        vocabulary_richness = len(unique_words)
        
        # 7. Number of solutions (answers)
        answer = row['answer']
        if isinstance(answer, list):
            # If answer is a list of strings, count elements
            num_solutions = len(answer)
            # Check if any answer contains multiple values (e.g., "$$24$$, $$36$$")
            solution_strings = []
            for a in answer:
                if isinstance(a, str):
                    # Look for multiple numbers separated by commas or other delimiters
                    solution_strings.extend(re.findall(r'\d+', a))
            solution_complexity = len(solution_strings)
        else:
            num_solutions = 1 if pd.notna(answer) else 0
            if isinstance(answer, str):
                solution_complexity = len(re.findall(r'\d+', answer))
            else:
                solution_complexity = 0
        
        # 8. Content length (character count)
        char_length = len(content)
        
        features.append({
            'question_id': idx,  # Use index as question ID
            'question_length': question_length,
            'num_variables': num_variables,
            'num_sentences': num_sentences,
            'num_clauses': num_clauses,
            'relies_on_image': relies_on_image,
            'vocabulary_richness': vocabulary_richness,
            'num_solutions': num_solutions,
            'solution_complexity': solution_complexity,
            'char_length': char_length
        })
    
    return pd.DataFrame(features)

# Apply to your question metadata
question_features = extract_question_features(question_metadata)

# Merge with existing question metadata
question_df_with_features = question_metadata.merge(
    question_features,
    left_index=True,
    right_on='question_id',
    how='left'
)

print("Feature extraction complete!")
print(question_features.head())
print("\nFeature statistics:")
print(question_features.describe())




Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\Hatem\AppData\Local\Temp\jieba.cache
Loading model cost 0.957 seconds.
Prefix dict has been built successfully.


Feature extraction complete!
   question_id  question_length  num_variables  num_sentences  num_clauses  \
0            0               37              0              0            7   
1            1               63              4              0            4   
2            2               57              3              0            2   
3            3               35              2              1            2   
4            4               37              0              0            4   

   relies_on_image  vocabulary_richness  num_solutions  solution_complexity  \
0                0                   30              1                    1   
1                0                   28              1                    1   
2                1                   38              1                    1   
3                1                   26              1                    1   
4                0                   32              1                    1   

   char_length  
0         

In [3]:
def extract_solution_features(question_df):
    """
    Extract linguistic and structural features from solution/analysis text.
    
    Features:
    - solution_length: number of words in solution (with repeats allowed)
    - num_equations: count of '=' signs in solution
    - num_steps: count of steps (based on : or 。 or , patterns)
    - solution_vocab: unique vocabulary words in solution
    - solution_relies_on_image: 1 if 'image' appears in solution
    - solution_complexity: count of unique math operations used (×, ÷, +, -, \square, etc.)
    """
    
    # Compile regex patterns
    equation_pattern = re.compile(r'=')  # Counts equations
    step_colon_pattern = re.compile(r'：')  # Chinese colon for step markers
    step_fullstop_pattern = re.compile(r'[。！？!?\.]')  # Sentence boundaries
    step_comma_pattern = re.compile(r'[，、；;]')  # Clause boundaries
    image_pattern = re.compile(r'image', re.IGNORECASE)
    
    # Math operation patterns
    multiply_pattern = re.compile(r'\\times')  # ×
    divide_pattern = re.compile(r'\\div')     # ÷
    square_pattern = re.compile(r'\\square')   # Square
    plus_pattern = re.compile(r'\+')          # +
    minus_pattern = re.compile(r'-')          # -
    
    features = []
    
    for idx, row in question_df.iterrows():
        analysis = str(row['analysis']) if pd.notna(row['analysis']) else ''
        
        # 1. Solution length (word count with repeats)
        words = list(jieba.cut(analysis))
        solution_length = len(words)
        
        # 2. Count equations (= signs)
        num_equations = len(equation_pattern.findall(analysis))
        
        # 3. Count steps
        # First try: count colons (Chinese step markers like "第一步：")
        colon_steps = len(step_colon_pattern.findall(analysis))
        
        # Second try: count sentences
        sentence_steps = len(step_fullstop_pattern.findall(analysis))
        
        # Third try: count clauses
        clause_steps = len(step_comma_pattern.findall(analysis))
        
        # Use the most reasonable step count (prefer colons, then sentences, then clauses)
        if colon_steps > 0:
            num_steps = colon_steps
        elif sentence_steps > 0:
            num_steps = sentence_steps
        else:
            num_steps = clause_steps
        
        # 4. Solution vocabulary richness (unique words)
        unique_words = set(jieba.cut(analysis))
        solution_vocab = len(unique_words)
        
        # 5. Check if relies on image
        solution_relies_on_image = 1 if image_pattern.search(analysis) else 0
        
        # 6. Solution complexity: count unique math operations used
        operations_used = set()
        
        if multiply_pattern.search(analysis):
            operations_used.add('multiplication')
        if divide_pattern.search(analysis):
            operations_used.add('division')
        if square_pattern.search(analysis):
            operations_used.add('square')
        if plus_pattern.search(analysis):
            operations_used.add('addition')
        if minus_pattern.search(analysis):
            operations_used.add('subtraction')
        
        # Also check for Chinese math words
        if '乘' in analysis or '×' in analysis:
            operations_used.add('multiplication')
        if '除' in analysis or '÷' in analysis:
            operations_used.add('division')
        if '加' in analysis or '+' in analysis:
            operations_used.add('addition')
        if '减' in analysis or '-' in analysis:
            operations_used.add('subtraction')
        if '平方' in analysis or '²' in analysis:
            operations_used.add('square')
        
        solution_complexity = len(operations_used)
        
        # 7. Additional: solution character length
        solution_char_length = len(analysis)
        
        # 8. Check if solution has multiple methods (contains words like "方法" or "另一种")
        has_multiple_methods = 1 if ('方法' in analysis and ('另' in analysis or '二' in analysis or '两' in analysis)) else 0
        
        features.append({
            'question_id': idx,
            'solution_length': solution_length,
            'num_equations': num_equations,
            'num_steps': num_steps,
            'solution_vocab': solution_vocab,
            'solution_relies_on_image': solution_relies_on_image,
            'solution_complexity': solution_complexity,
            'solution_char_length': solution_char_length,
            'has_multiple_methods': has_multiple_methods,
            'operations_used': ', '.join(operations_used) if operations_used else 'none'
        })
    
    return pd.DataFrame(features)

# Apply to your question metadata
solution_features = extract_solution_features(question_metadata)

# Merge with existing question metadata
question_df_with_features = question_df_with_features.merge(
    solution_features,
    left_index=True,
    right_on='question_id',
    how='left'
)

print("Solution feature extraction complete!")
print(solution_features.head())
print("\nFeature statistics:")
print(solution_features.describe())

# Check a sample
sample_idx = 0
print(f"\nSample Question {sample_idx}:")
print(f"Analysis text: {question_metadata.iloc[sample_idx]['analysis'][:200]}...")
print(f"Solution features: {solution_features.iloc[sample_idx].to_dict()}")




<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Hatem\AppData\Local\Temp\ipykernel_9696\719468575.py:2: SyntaxWarning: invalid escape sequence '\s'
  """


Solution feature extraction complete!
   question_id  solution_length  num_equations  num_steps  solution_vocab  \
0            0               51              1          6              23   
1            1              118              4          7              36   
2            2               77              1          5              27   
3            3              152              1          3              45   
4            4               17              1          0               9   

   solution_relies_on_image  solution_complexity  solution_char_length  \
0                         0                    1                    73   
1                         0                    2                   150   
2                         0                    1                   103   
3                         0                    1                   197   
4                         0                    1                    26   

   has_multiple_methods           operations_used  
0 

In [4]:
question_df_with_features.to_csv('question_metadata_with_all_features.csv', index=False, encoding='utf-8-sig')

## Higher Level Topic Classification

In [4]:
def map_kc_routes_to_super_topic(kc_routes_list):
    """
    Extract the first two nodes from each KC path and map to a super topic ID.
    Example: '拓展思维----计数模块----加乘原理----加乘原理综合' 
             → super topic: '拓展思维----计数模块'
             → mapped to a single ID
    """
    # If it's a string and contains '----', treat it as a single path
    if isinstance(kc_routes_list, str):
        kc_routes_list = [kc_routes_list]
    elif not isinstance(kc_routes_list, list):
        return kc_routes_list
    
    # Process each path
    super_topic_ids = []
    for full_path in kc_routes_list:
        if '----' in full_path:
            parts = full_path.split('----')
            # Take first two nodes
            super_topic = '----'.join(parts[:2]) if len(parts) >= 2 else full_path
        else:
            super_topic = full_path
        
        # Map to ID, or keep original if not found
        if(super_topic_to_id.get(super_topic, f"NOT FOUND: {super_topic}") not in super_topic_ids):
            super_topic_ids.append(super_topic_to_id.get(super_topic, f"NOT FOUND: {super_topic}"))
    
    return super_topic_ids

# ============================================================
# Build the super topic mapping
# ============================================================

# First, extract all unique super topics from your data
all_super_topics = set()

for routes in question_df['kc_routes']:
    if isinstance(routes, list):
        for path in routes:
            if '----' in path:
                parts = path.split('----')
                if len(parts) >= 2:
                    super_topic = '----'.join(parts[:2])
                    all_super_topics.add(super_topic)

# Create mapping from super topic to ID
super_topic_to_id = {topic: idx for idx, topic in enumerate(sorted(all_super_topics))}

print(f"Found {len(super_topic_to_id)} unique super topics")
print("Sample super topics:")
for topic in list(super_topic_to_id.items())[:10]:
    print(f"  {topic}")

# ============================================================
# Apply to your dataframe
# ============================================================

# Add super topic IDs to question_df
question_df['super_topic_ids'] = question_df['kc_routes'].apply(map_kc_routes_to_super_topic)

# Add the actual super topic names for reference
def get_super_topic_names(kc_routes_list):
    """Get the super topic names (first two nodes) for each path"""
    if isinstance(kc_routes_list, str):
        kc_routes_list = [kc_routes_list]
    elif not isinstance(kc_routes_list, list):
        return kc_routes_list
    
    super_topics = []
    for full_path in kc_routes_list:
        if '----' in full_path:
            parts = full_path.split('----')
            super_topic = '----'.join(parts[:2]) if len(parts) >= 2 else full_path
        else:
            super_topic = full_path
        super_topics.append(super_topic)
    
    return super_topics

question_df['super_topic_names'] = question_df['kc_routes'].apply(get_super_topic_names)

# ============================================================
# Check distribution
# ============================================================

# Flatten all super topic IDs for counting
all_super_ids = []
for ids in question_df['super_topic_ids']:
    if isinstance(ids, list):
        all_super_ids.extend(ids)

from collections import Counter
super_topic_counts = Counter(all_super_ids)

print("\nSuper topic distribution:")
for topic_id, count in super_topic_counts.most_common(10):
    # Find the topic name for this ID
    topic_name = [k for k, v in super_topic_to_id.items() if v == topic_id]
    print(f"  {topic_id}: {topic_name[0] if topic_name else 'Unknown'} ({count} questions)")


print("\nSample questions with super topics:")
print(question_df[['content', 'super_topic_names', 'super_topic_ids']].head(10))

Found 61 unique super topics
Sample super topics:
  ('七大能力----图形认知', 0)
  ('七大能力----实践应用', 1)
  ('七大能力----运算求解', 2)
  ('七大能力----逻辑分析', 3)
  ('学习能力----七大能力', 4)
  ('思想----分类讨论思想', 5)
  ('思想----枚举思想', 6)
  ('思想----逆向思想', 7)
  ('拓展思维----几何模块', 8)
  ('拓展思维----应用题模块', 9)

Super topic distribution:
  9: 拓展思维----应用题模块 (2344 questions)
  14: 拓展思维----计算模块 (1271 questions)
  13: 拓展思维----计数模块 (1141 questions)
  11: 拓展思维----组合模块 (1034 questions)
  8: 拓展思维----几何模块 (730 questions)
  26: 知识点----应用题模块 (400 questions)
  58: 课内题型----综合与实践 (363 questions)
  10: 拓展思维----数论模块 (287 questions)
  39: 知识点----计算模块 (265 questions)
  12: 拓展思维----行程模块 (211 questions)

Sample questions with super topics:
                                             content  \
0  学校有舞蹈，唱歌、围棋、绘画四种兴趣班． 小宇、小明、小丽三个小朋友准备报名，每人只能报一个...   
1  书架上有 $$2$$ 本不同的英语书，$$4$$ 本不同的语文书，$$3$$ 本不同的数学书...   
2  用$$5$$种不同的颜色给下面的图形染色，要求相邻的区域（有公共边的两区域称为相邻）染成不同...   
3  用四种颜色去涂如图所示的三块区域，要求相邻的区域涂不同的颜色，那么共有种不同的涂法.\n q...   
4  按下表给出的词造句，每句必须包括一个人、一个交通工具，以及一个目的

In [12]:
#load the question metadata and add the super topic features to it
question_metadata=pd.read_csv(os.getenv("question_text_features"))

question_metadata=pd.merge(
 question_metadata,question_df,left_on='question_id',right_on='index',how='left'   
)

question_metadata.drop(
    columns=[
        'index_x', 'content_x', 'kc_routes_x', 'answer_x', 'analysis_x',
        'type_x', 'options_x', 'question_id_x',
        'index_y', 'content_y', 'kc_routes_y', 'answer_y', 'analysis_y',
        'type_y', 'options_y', 'question_id_y'
    ],
    inplace=True
)




In [13]:
print(question_metadata.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7652 entries, 0 to 7651
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   question_id               7652 non-null   int64  
 1   kc_ids                    7652 non-null   object 
 2   attempted                 7618 non-null   float64
 3   correct                   7618 non-null   float64
 4   wrong                     7618 non-null   float64
 5   error_rate                7618 non-null   float64
 6   question_length           7652 non-null   int64  
 7   num_variables             7652 non-null   int64  
 8   num_sentences             7652 non-null   int64  
 9   num_clauses               7652 non-null   int64  
 10  relies_on_image           7652 non-null   int64  
 11  vocabulary_richness       7652 non-null   int64  
 12  num_solutions             7652 non-null   int64  
 13  solution_complexity_x     7652 non-null   int64  
 14  char_len

In [14]:
question_metadata.to_csv('question_metadata.csv',index=False,encoding='utf-8')

In [22]:
kc_supertopic_map = []

for idx, row in question_metadata.iterrows():
    kc_ids = row['kc_ids']  # List of KC IDs
    super_topics = row['super_topic_ids']  # List of super topic IDs
    
    # Handle if they're strings (might need eval)
    if isinstance(kc_ids, str):
        kc_ids = eval(kc_ids)
    if isinstance(super_topics, str):
        super_topics = eval(super_topics)
    
    # Ensure both are lists and same length
    if isinstance(kc_ids, list) and isinstance(super_topics, list):
        for kc_id, super_topic in zip(kc_ids, super_topics):
            kc_supertopic_map.append({
                'kc_id': kc_id,
                'super_topic_id': super_topic
            })

# Convert to DataFrame
kc_supertopic_df = pd.DataFrame(kc_supertopic_map)

# Remove duplicates (keep first occurrence)
kc_supertopic_df = kc_supertopic_df.drop_duplicates(subset=['kc_id'])

print(f"Found {len(kc_supertopic_df)} unique KC-super topic mappings")
print(kc_supertopic_df.head())

kc_supertopic_df.to_csv('super_topic_map.csv',index=False)

Found 839 unique KC-super topic mappings
   kc_id  super_topic_id
0      0              13
1      1              13
2      3              13
4      4              13
6      2              13
